In [1]:
import h5py
import numpy as np
from sympy import Matrix
import math

### Number of objects

## Reps for new images

In [12]:
f1 = h5py.File('demo3D50_output/demo3D50.mat')
y1 = f1['model']['cellShapeModel']['all_spharm_descriptors']
z1 = np.array(y1)
z1.shape

(1049, 3, 1024)

In [17]:
p8_reps = z1[:252, :, :]
p24_reps = z1[252:(252+245), :, :]
p34_reps = z1[(252+245):(252+245+266), :, :]
p52_reps = z1[(252+245+266):, :, :]

print(p8_reps.shape)
print(p24_reps.shape)
print(p34_reps.shape)
print(p52_reps.shape)

(252, 3, 1024)
(245, 3, 1024)
(266, 3, 1024)
(286, 3, 1024)


### Converting from 3D to 2D

In [18]:
def get_3D_from_2D(mat):
    x,y,z = mat.shape
    y_len = y*z*2
    shapes = np.zeros(shape=(x,y_len))
    for i in range(x):
        for j in range(y):
            for k in range(z):
                a,b = mat[i][j][k]
                shapes[i][j*z + k] = a
                shapes[i][y*z + j*z + k] = b
    num_zeros1 =0 
    num_zeros2 = 0
    sum1 = 0
    sum2 = 0
    for i in range(x):
        for j in range(y_len):
            if(shapes[i][j] == 0):
                num_zeros1 +=1
            sum1 += shapes[i][j]

    for i in range(x):
        for j in range(y):
            for k in range(z):
                a,b = mat[i][j][k]
                sum2 += a
                sum2 += b
                if(a==0):
                    num_zeros2 +=1
                if(b==0):
                    num_zeros2 +=1
    print(num_zeros1, num_zeros2, sum1, sum2)
    return shapes

In [19]:
p8_reps_2D = get_3D_from_2D(p8_reps)
p24_reps_2D = get_3D_from_2D(p24_reps)
p34_reps_2D = get_3D_from_2D(p34_reps)
p52_reps_2D = get_3D_from_2D(p52_reps)

print(p8_reps_2D.shape, p24_reps_2D.shape, p34_reps_2D.shape, p52_reps_2D.shape)

1191616 1191616 47091.75352930948 47091.75352930962
1196691 1196691 47077.23562556858 47077.23562556857
1316790 1316790 49360.56814585474 49360.5681458548
1440695 1440695 51815.831616493924 51815.831616494
(252, 6144) (245, 6144) (266, 6144) (286, 6144)


In [20]:
np.save("p8_ShapeReps_252x6144", p8_reps_2D)
np.save("p24_ShapeReps_245x6144", p24_reps_2D)
np.save("p34_ShapeReps_266x6144", p34_reps_2D)
np.save("p52_ShapeReps_286x6144", p52_reps_2D)

## Computing Distance matrices

In [23]:
def get_sqdist(vec1, vec2):
    length = len(vec1)
    assert(len(vec2)==length)
    sum_sq = 0
    for i in range(length):
        sum_sq += (vec1[i]-vec2[i])**2
    return sum_sq

def get_absdist(vec1, vec2):
    length = len(vec1)
    assert(len(vec2)==length)
    sum_abs = 0
    for i in range(length):
        sum_abs += abs(vec1[i]-vec2[i])
    return sum_abs

def get_distanceMat_from_2D(name, dist="euc"):
    shapes = np.load(name)
    distance = np.zeros((len(shapes),len(shapes)))
    for i in range(len(shapes)):
        for j in range(len(shapes)):
            if(dist == "euc"):
                a = get_sqdist(shapes[i], shapes[j])
            elif(dist == "man"):
                a = get_absdist(shapes[i], shapes[j])
            else:
                print("Invalid choice! Choose between euc and man")
                return -1
            distance[i][j] = a
    print(distance[34][23], distance[23][34], distance[4][4])
    return distance

In [25]:
p8_euc = get_distanceMat_from_2D('p8_ShapeReps_252x6144.npy', dist="euc")
p8_man = get_distanceMat_from_2D('p8_ShapeReps_252x6144.npy', dist="man")

np.save("p8_ShapeDist_6144_Euc", p8_euc)
np.save("p8_ShapeDist_6144_Man", p8_man)

p24_euc = get_distanceMat_from_2D('p24_ShapeReps_245x6144.npy', dist="euc")
p24_man = get_distanceMat_from_2D('p24_ShapeReps_245x6144.npy', dist="man")

np.save("p24_ShapeDist_6144_Euc", p24_euc)
np.save("p24_ShapeDist_6144_Man", p24_man)

p34_euc = get_distanceMat_from_2D('p34_ShapeReps_266x6144.npy', dist="euc")
p34_man = get_distanceMat_from_2D('p34_ShapeReps_266x6144.npy', dist="man")

np.save("p34_ShapeDist_6144_Euc", p34_euc)
np.save("p34_ShapeDist_6144_Man", p34_man)

p52_euc = get_distanceMat_from_2D('p52_ShapeReps_286x6144.npy', dist="euc")
p52_man = get_distanceMat_from_2D('p52_ShapeReps_286x6144.npy', dist="man")

np.save("p52_ShapeDist_6144_Euc", p52_euc)
np.save("p52_ShapeDist_6144_Man", p52_man)

438.6399327509479 438.6399327509479 0.0
190.19711781386627 190.19711781386627 0.0
541.2904458431634 541.2904458431634 0.0
152.73821272359788 152.73821272359788 0.0
94.17912656892771 94.17912656892771 0.0
112.6818466312678 112.6818466312678 0.0
888.732796590144 888.732796590144 0.0
186.5994815184998 186.5994815184998 0.0
